# Lesson 11: PyTorch 2.0 - Speed and New Features

This notebook covers:
- PyTorch 2.0's major performance improvements
- `torch.compile()` - 1 line for faster training!
- New device management features
- TensorFloat32 (TF32) for faster GPU computing
- Performance benchmarking
- Real-world speedups

## What's New in PyTorch 2.0?

PyTorch 2.0 is a **major release** focused on **performance**.

### Key Innovation: torch.compile()

**One line of code for faster training:**
```python
model = torch.compile(model)
```

**Typical speedups:**
- Training: 30-50% faster
- Inference: 20-40% faster
- Memory: Often reduced

### How torch.compile() Works:

1. **Graph capture**: Records model operations
2. **Optimization**: Fuses operations, removes overhead
3. **Code generation**: Creates optimized kernels
4. **Execution**: Runs faster compiled code

**Benefits:**
- No model changes needed!
- Backward compatible
- Automatic optimization
- Works with existing code

## Other PyTorch 2.0 Features:

### 1. Better Device Management
```python
# Context manager (temporary)
with torch.device('cuda'):
    layer = nn.Linear(10, 10)  # Created on GPU

# Global default (persistent)
torch.set_default_device('cuda')
layer = nn.Linear(10, 10)  # Created on GPU
```

### 2. TensorFloat32 (TF32)
- Faster matrix operations on new GPUs (Ampere+)
- Automatic mixed precision
- Enabled with: `torch.backends.cuda.matmul.allow_tf32 = True`

### 3. Better Error Messages
- Clearer debugging information
- More helpful stack traces

## What We'll Benchmark:

- **Model**: ResNet50 (25M parameters)
- **Dataset**: CIFAR-10 (60,000 images)
- **Comparison**: PyTorch 1.x vs 2.0 with compile()
- **Metrics**: Training time, throughput, memory

## Setup and Version Check

In [ ]:
# Check Python version
import sys
sys.version

In [ ]:
# Check PyTorch version
# IMPORTANT: Need PyTorch 2.0+ for torch.compile()
import torch
import torchvision

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print()
print(f"TorchVision version: {torchvision.__version__}")

# Verify PyTorch 2.0+
if int(torch.__version__.split('.')[0]) < 2:
    print("\n⚠️  WARNING: PyTorch 2.0+ required for torch.compile()")
else:
    print(f"\n✓ PyTorch {torch.__version__} ready!")

In [ ]:
# Quick test of torch.compile()
# Uncomment to test:
# model = torchvision.models.resnet50()
# compiled_model = torch.compile(model)
# print("✓ torch.compile() works!")

## Part 1: New Device Management Features

PyTorch 2.0 makes device management easier!

In [ ]:
# Check GPU info (uncomment to run)
# !nvidia-smi

In [ ]:
# Get GPU information
if torch.cuda.is_available():
    # Run nvidia-smi and check for issues
    gpu_info = !nvidia-smi
    gpu_info = '\n'.join(gpu_info)
    
    if gpu_info.find("failed") >= 0:
        print("Not connected to a GPU.")
        print("To leverage PyTorch 2.0 best, connect to a GPU.")
    
    # Get GPU name
    if "NVIDIA" in gpu_info:
        # Extract GPU model name
        for line in gpu_info.split('\n'):
            if 'NVIDIA' in line:
                gpu_name = line.split('NVIDIA')[1].split('|')[0].strip()
                print(f"GPU: NVIDIA {gpu_name}")
                break
    
    # Determine GPU compute capability
    # This affects which optimizations are available
    gpu_props = torch.cuda.get_device_properties(0)
    GPU_SCORE = (gpu_props.major, gpu_props.minor)
    print(f"GPU Compute Capability: {GPU_SCORE[0]}.{GPU_SCORE[1]}")
    
    # Ampere (8.0+) and newer support TF32
    if GPU_SCORE >= (8, 0):
        print("✓ GPU supports TensorFloat32 (TF32) - faster training!")
    else:
        print("  GPU doesn't support TF32 (need Ampere+ / 8.0+)")
else:
    print("No GPU available, using CPU")
    GPU_SCORE = (0, 0)

### Feature 1: Device Context Manager

In [ ]:
# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"

# NEW in PyTorch 2.0: Device context manager
# All tensors created in this block will be on the specified device
with torch.device(device):
    layer = torch.nn.Linear(20, 30)
    print(f"Layer weights device: {layer.weight.device}")
    print(f"✓ Created on {device} automatically!")

# Outside the context, default device is CPU
layer_cpu = torch.nn.Linear(20, 30)
print(f"\nOutside context, device: {layer_cpu.weight.device}")

### Feature 2: Global Default Device

In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# NEW in PyTorch 2.0: Set global default device
# All tensors created will be on this device by default
torch.set_default_device(device)

# Now all tensors are automatically on the default device!
layer = torch.nn.Linear(20, 30)
print(f"Layer weights device: {layer.weight.device}")

tensor = torch.randn(3, 3)
print(f"Tensor device: {tensor.device}")

print(f"\n✓ No more manual .to(device) calls!")

# Note: This persists for the entire session
# Reset with: torch.set_default_device('cpu')

## Part 2: Setup Model and Data

We'll use ResNet50 on CIFAR-10 for benchmarking.

In [ ]:
# Print versions and device
print(f"PyTorch version: {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")

# Set target device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### Create Model

In [ ]:
# Create ResNet50 with pretrained ImageNet weights
model_weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
transforms = model_weights.transforms()

# Create model
model = torchvision.models.resnet50(weights=model_weights)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")

In [ ]:
def create_model(num_classes=10):
    """
    Creates a ResNet50 model with latest weights and transforms.
    
    Args:
        num_classes: Number of output classes (CIFAR-10 has 10)
        
    Returns:
        tuple: (model, transforms)
    """
    # Get pretrained weights and transforms
    model_weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
    transforms = model_weights.transforms()
    
    # Create model
    model = torchvision.models.resnet50(weights=model_weights)
    
    # Modify classifier for CIFAR-10 (10 classes vs ImageNet's 1000)
    model.fc = torch.nn.Linear(in_features=2048, out_features=num_classes)
    
    return model, transforms

### Configure Batch Size Based on GPU Memory

In [ ]:
# Check available GPU memory
if torch.cuda.is_available():
    total_free_gpu_memory, total_gpu_memory = torch.cuda.mem_get_info()
    print(f"Free GPU memory: {round(total_free_gpu_memory * 1e-9, 3)} GB")
    print(f"Total GPU memory: {round(total_gpu_memory * 1e-9, 3)} GB")
else:
    total_free_gpu_memory = 0
    print("No GPU available")

In [ ]:
# Set batch size based on available GPU memory
# Larger batches = faster training (up to a point)
total_free_gpu_memory_gb = round(total_free_gpu_memory * 1e-9, 3)

if total_free_gpu_memory_gb >= 16:
    BATCH_SIZE = 128  # Large GPU (A100, V100)
    IMAGE_SIZE = 224
    print(f"GPU memory {total_free_gpu_memory_gb}GB >= 16GB")
    print(f"Using BATCH_SIZE={BATCH_SIZE}, IMAGE_SIZE={IMAGE_SIZE}")
    
elif total_free_gpu_memory_gb >= 8:
    BATCH_SIZE = 64   # Medium GPU (RTX 3080, T4)
    IMAGE_SIZE = 224
    print(f"GPU memory {total_free_gpu_memory_gb}GB >= 8GB")
    print(f"Using BATCH_SIZE={BATCH_SIZE}, IMAGE_SIZE={IMAGE_SIZE}")
    
else:
    BATCH_SIZE = 32   # Small GPU or CPU
    IMAGE_SIZE = 224
    print(f"GPU memory < 8GB or using CPU")
    print(f"Using BATCH_SIZE={BATCH_SIZE}, IMAGE_SIZE={IMAGE_SIZE}")

print(f"\nFinal config: BATCH_SIZE={BATCH_SIZE}, IMAGE_SIZE={IMAGE_SIZE}")

In [ ]:
# Update transforms with our image size
transforms.crop_size = IMAGE_SIZE
transforms.resize_size = IMAGE_SIZE
print(f"Updated data transforms:\n{transforms}")

### Enable TensorFloat32 (TF32) on Supported GPUs

**What is TF32?**
- New data format on Ampere+ GPUs (RTX 30/40 series, A100)
- Faster than FP32, similar accuracy
- Automatic mixed precision
- ~8x speedup for matrix operations

**When to use:**
- GPU compute capability >= 8.0 (Ampere or newer)
- Training large models
- Matrix-heavy operations

In [ ]:
# Enable TF32 if supported
if GPU_SCORE >= (8, 0):
    print(f"[INFO] GPU score {GPU_SCORE} supports TensorFloat32 (TF32)")
    print("Enabling TF32 for faster training on Ampere+ GPUs")
    
    # Enable TF32 for matrix multiplications
    torch.backends.cuda.matmul.allow_tf32 = True
    
    # Enable TF32 for cuDNN
    torch.backends.cudnn.allow_tf32 = True
    
    print("✓ TF32 enabled - expect ~20-40% speedup!")
else:
    print(f"[INFO] GPU score {GPU_SCORE} doesn't support TF32")
    print("TF32 requires Ampere or newer (compute capability >= 8.0)")
    print("Examples: RTX 3000/4000 series, A100, H100")

## Part 3: Setup Data

Using CIFAR-10 for benchmarking:
- 60,000 images (50k train, 10k test)
- 10 classes
- 32×32 pixels (we'll resize to 224×224)

In [ ]:
# Create generator for reproducibility
device_obj = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = torch.Generator(device=device_obj)

# Create datasets
# CIFAR-10 will download automatically on first run
train_dataset = torchvision.datasets.CIFAR10(
    root='.',
    train=True,
    download=True,
    transform=transforms
)

test_dataset = torchvision.datasets.CIFAR10(
    root='.',
    train=False,
    download=True,
    transform=transforms
)

print(f"Training samples: {len(train_dataset):,}")
print(f"Testing samples: {len(test_dataset):,}")
print(f"Classes: {train_dataset.classes}")

### Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader
import os

In [ ]:
# Use all CPU cores for data loading
NUM_WORKERS = os.cpu_count()

# Create DataLoaders
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=generator,
    pin_memory=True  # Faster GPU transfer
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
# Print details
print(f"Train dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f"Test dataloader: {len(test_dataloader)} batches of {BATCH_SIZE}")
print(f"Number of workers: {NUM_WORKERS}")
print(f"\nNote: More workers = faster data loading (usually)")
print(f"But too many workers can slow things down!")

## Part 4: Training Functions

In [ ]:
import time
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

In [ ]:
def train_step(
    epoch: int,
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    disable_progress_bar: bool = False
) -> Tuple[float, float]:
    """
    Performs one training epoch.
    
    Returns:
        tuple: (train_loss, train_accuracy)
    """
    model.train()
    
    train_loss, train_acc = 0, 0
    
    # Training loop with progress bar
    for batch, (X, y) in enumerate(tqdm(
        dataloader,
        desc=f"Epoch {epoch} Training",
        disable=disable_progress_bar
    )):
        # Move data to device
        X, y = X.to(device), y.to(device)
        
        # Forward pass
        y_pred = model(X)
        
        # Calculate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)
    
    # Average loss and accuracy
    train_loss /= len(dataloader)
    train_acc /= len(dataloader)
    
    return train_loss, train_acc

In [ ]:
def test_step(
    epoch: int,
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    device: torch.device,
    disable_progress_bar: bool = False
) -> Tuple[float, float]:
    """
    Performs one testing epoch.
    
    Returns:
        tuple: (test_loss, test_accuracy)
    """
    model.eval()
    
    test_loss, test_acc = 0, 0
    
    # Testing loop with progress bar
    with torch.inference_mode():
        for batch, (X, y) in enumerate(tqdm(
            dataloader,
            desc=f"Epoch {epoch} Testing",
            disable=disable_progress_bar
        )):
            # Move data to device
            X, y = X.to(device), y.to(device)
            
            # Forward pass
            test_pred_logits = model(X)
            
            # Calculate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            
            # Calculate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += (test_pred_labels == y).sum().item() / len(test_pred_labels)
    
    # Average loss and accuracy
    test_loss /= len(dataloader)
    test_acc /= len(dataloader)
    
    return test_loss, test_acc

In [ ]:
def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    epochs: int,
    device: torch.device
) -> Dict[str, List]:
    """
    Trains model for specified epochs and tracks metrics.
    
    Returns:
        dict: Training results including losses, accuracies, and timing
    """
    # Initialize results dictionary
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
        "epoch_times": []
    }
    
    # Training loop
    for epoch in range(epochs):
        # Time the epoch
        epoch_start = time.time()
        
        # Train and test
        train_loss, train_acc = train_step(
            epoch=epoch,
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )
        
        test_loss, test_acc = test_step(
            epoch=epoch,
            model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device
        )
        
        epoch_end = time.time()
        epoch_time = epoch_end - epoch_start
        
        # Print results
        print(
            f"Epoch: {epoch} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Test Loss: {test_loss:.4f} | "
            f"Test Acc: {test_acc:.4f} | "
            f"Time: {epoch_time:.2f}s"
        )
        
        # Update results
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
        results["epoch_times"].append(epoch_time)
    
    return results

## Part 5: Benchmark WITHOUT torch.compile()

First, let's train the model the traditional way (PyTorch 1.x style).

In [ ]:
# Training hyperparameters
NUM_EPOCHS = 5
LEARNING_RATE = 0.003

print(f"Training for {NUM_EPOCHS} epochs")
print(f"Learning rate: {LEARNING_RATE}")

In [ ]:
device

In [ ]:
# Create model WITHOUT compile
print("[BASELINE] Training WITHOUT torch.compile()...\n")

model, transforms = create_model()
model.to(device)

# Create loss and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Train and track time
start_time = time.time()

single_run_results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS,
    device=device
)

end_time = time.time()
total_time_baseline = end_time - start_time

print(f"\n{'='*70}")
print(f"BASELINE (no compile) Total Time: {total_time_baseline:.2f}s")
print(f"Average time per epoch: {total_time_baseline/NUM_EPOCHS:.2f}s")
print(f"{'='*70}")

## Part 6: Benchmark WITH torch.compile()

Now let's use PyTorch 2.0's **torch.compile()** for faster training!

In [ ]:
# Suppress compilation warnings
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [ ]:
# Create model WITH compile
print("[PYTORCH 2.0] Training WITH torch.compile()...\n")

model, transforms = create_model()
model.to(device)

# Create loss and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ✨ THE MAGIC LINE ✨
# Compile the model for faster execution
# This analyzes the model and generates optimized code
model = torch.compile(model)
print("✓ Model compiled!\n")

# Train and track time
start_time = time.time()

compile_run_results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS,
    device=device
)

end_time = time.time()
total_time_compiled = end_time - start_time

print(f"\n{'='*70}")
print(f"COMPILED (torch.compile) Total Time: {total_time_compiled:.2f}s")
print(f"Average time per epoch: {total_time_compiled/NUM_EPOCHS:.2f}s")
print(f"{'='*70}")

## Part 7: Compare Results

Let's see how much faster torch.compile() made training!

In [ ]:
# Calculate speedup
speedup = total_time_baseline / total_time_compiled
time_saved = total_time_baseline - total_time_compiled
percent_faster = ((total_time_baseline - total_time_compiled) / total_time_baseline) * 100

print("\n" + "="*70)
print("PERFORMANCE COMPARISON")
print("="*70)

print(f"\nBaseline (no compile):    {total_time_baseline:.2f}s")
print(f"Compiled (torch.compile): {total_time_compiled:.2f}s")

print(f"\n🚀 Speedup: {speedup:.2f}x faster")
print(f"⏱️  Time saved: {time_saved:.2f}s ({percent_faster:.1f}% faster)")

if speedup > 1.3:
    print(f"\n✅ Excellent speedup with torch.compile()!")
elif speedup > 1.1:
    print(f"\n✓ Good speedup with torch.compile()")
else:
    print(f"\n⚠️ Modest speedup - may vary by GPU/model")

print("\nNote: First epoch with compile() is slower (compilation overhead)")
print("Subsequent epochs benefit from optimized code!")
print("="*70)

## Summary: PyTorch 2.0 Benefits

### Key Takeaways:

**1. torch.compile() - The Game Changer**
```python
model = torch.compile(model)  # One line!
```

**Typical Results:**
- Training: 30-50% faster
- Inference: 20-40% faster
- No code changes needed!
- Backward compatible

**2. Better Device Management**
```python
# Context manager
with torch.device('cuda'):
    model = MyModel()

# Global default
torch.set_default_device('cuda')
```

**3. TensorFloat32 (TF32)**
```python
torch.backends.cuda.matmul.allow_tf32 = True
# ~20-40% speedup on Ampere+ GPUs
```

### When torch.compile() Helps Most:

✅ **Large models**: More operations to optimize
✅ **GPU training**: Kernel fusion, memory optimization
✅ **Production inference**: Consistent speedups
✅ **Repeated operations**: Compilation overhead amortized

### When torch.compile() Helps Less:

❌ **First run**: Compilation overhead
❌ **Small models**: Less to optimize
❌ **CPU only**: GPU optimizations don't apply
❌ **Custom CUDA kernels**: Already optimized

### Production Deployment:

**For Training:**
```python
model = torch.compile(model)
# Train as normal - 30-50% faster!
```

**For Inference:**
```python
model = torch.compile(model, mode="reduce-overhead")
# Even faster for inference
```

**Modes:**
- `default`: Balanced (recommended)
- `reduce-overhead`: Faster inference
- `max-autotune`: Maximum optimization (slower compile)

### Real-World Impact:

**Example: ResNet50 on CIFAR-10**
- Baseline: 200s for 5 epochs
- Compiled: 140s for 5 epochs
- Speedup: 1.43x (43% faster)

**Scaling to 100 epochs:**
- Baseline: 4,000s (~66 minutes)
- Compiled: 2,800s (~47 minutes)
- **Save 19 minutes!**

### Best Practices:

1. **Always try torch.compile()** - It's one line!
2. **Benchmark your specific case** - Results vary
3. **Use TF32 on Ampere+** - Free speedup
4. **Profile before/after** - Measure improvements
5. **Watch first epoch** - Expect compilation overhead

### Migration Guide:

**From PyTorch 1.x to 2.0:**
```python
# Old code (still works!)
model = MyModel()
model.to(device)
# train...

# New code (faster!)
model = MyModel()
model.to(device)
model = torch.compile(model)  # Add this line
# train... (30-50% faster)
```

**That's it!** PyTorch 2.0 is backward compatible.

### Future of PyTorch:

- **torch.compile()** will get faster
- More backends (CPU, mobile, custom hardware)
- Better error messages
- Easier deployment

**Bottom line:** PyTorch 2.0 makes your code faster with minimal effort!